# 🚀 Night Loom Engine — Google Colab Free T4 GPU Microservice Worker
This notebook turns a free Google Colab T4 GPU instance into a remote AI microservice endpoint for **SDXL-Turbo Image Generation** and **SadTalker Avatar Video Rendering**.

In [ ]:
# Step 1: Install PyTorch CUDA, Diffusers, FastAPI, and Tunneling
!pip install -q diffusers transformers accelerate torch torchvision safetensors pyngrok fastapi uvicorn pydantic python-multipart

In [ ]:
# Step 2: Clone YT-Automation-Hosted Repository
import os
# If your repo is Private, paste your GitHub Personal Access Token (PAT) below:
GH_TOKEN = ""

if GH_TOKEN:
    clone_url = f"https://{GH_TOKEN}@github.com/1919-14/YT-Automation-Hosted.git"
else:
    clone_url = "https://github.com/1919-14/YT-Automation-Hosted.git"

!git clone {clone_url} /content/YT-Automation-Hosted
%cd /content/YT-Automation-Hosted

In [ ]:
# Step 3: Launch FastAPI GPU Microservice Server
import os
import sys
import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok

# 🔑 Paste your free ngrok Authtoken below (Get it from https://dashboard.ngrok.com/get-started/your-authtoken):
NGROK_AUTHTOKEN = ""

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

app = FastAPI(title="Night Loom GPU Worker API")

class SDXLRequest(BaseModel):
    prompts: list[str]
    video_id: int
    style: str = "dark fantasy"

@app.get("/health")
def healthcheck():
    return {
        "status": "online",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "vram_free_gb": round(torch.cuda.mem_get_info()[0] / 1024**3, 2) if torch.cuda.is_available() else 0
    }

PORT = 8000
if not NGROK_AUTHTOKEN:
    print("❌ NGROK_AUTHTOKEN IS MISSING!")
    print("👉 Get your free token in 10 seconds here: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("👉 Paste it into NGROK_AUTHTOKEN = 'your_token_here' above and re-run cell 3.")
else:
    public_url = ngrok.connect(PORT).public_url
    print("\n" + "="*60)
    print("⚡ GPU WORKER IS ONLINE!")
    print("🔑 Copy this GPU_WORKER_URL into your Hugging Face Space Secrets:")
    print(f"👉 GPU_WORKER_URL = {public_url}")
    print("="*60 + "\n")
    uvicorn.run(app, host="0.0.0.0", port=PORT)
